# **Simple Architecture**

In [ ]:
from keras.models import Model

In [ ]:
model = Model(inputs = x ,outputs = [output1,output2])

In [ ]:
from keras.layers import *

In [ ]:
x = Input(shape=(3,))

In [ ]:
hidden1=Dense(128,activation='relu')(x)
hidden2=Dense(64,activation='relu')(hidden1)
output1=Dense(1,name='output1')(hidden2)
output2=Dense(1,name='output2')(hidden2)

In [ ]:
model.summary()

In [ ]:
from keras.utils import plot_model
plot_model(model,show_shapes=True)

# **Complex Architecture**

In [ ]:
inputA = Input(shape=(32,))
inputB = Input(shape=(128,))

In [ ]:
x = Dense(8, activation="relu")(inputA)
x1 = Dense(4, activation="relu")(x)

In [ ]:
y = Dense(64, activation="relu")(inputB)
y1 = Dense(32, activation="relu")(y)
y2 = Dense(4, activation="relu")(y1)

In [ ]:
combined = concatenate([x1, y2])

In [ ]:
z = Dense(2, activation="relu")(combined)
z1 = Dense(1, activation="linear")(z)

In [ ]:
model = Model(inputs=[inputA, inputB], outputs=z1)

In [ ]:
from keras.utils import plot_model
plot_model(model)

#**Multiple Output Model**
age and gender prediction model project utk face dataset

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("jangedoo/utkface-new")

print("Path to dataset files:", path)

In [ ]:
import os
import numpy as np
import pandas as pd
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
folder_path = '/kaggle/input/utkface-new/UTKFace'

In [ ]:
age=[]
gender=[]
img_path=[]
for file in os.listdir(folder_path):
  age.append(int(file.split('_')[0]))
  gender.append(int(file.split('_')[1]))
  img_path.append(file)

In [ ]:
len(age)

In [ ]:
df = pd.DataFrame({'age':age,'gender':gender,'img':img_path})

In [ ]:
df

In [ ]:
df.shape

In [ ]:
train_df = df.sample(frac=1,random_state=0).iloc[:20000]
test_df = df.sample(frac=1,random_state=0).iloc[20000:]

In [ ]:
test_df.shape

In [ ]:
train_df.shape

In [ ]:
train_datagen = ImageDataGenerator(rescale=1./255,
                                   rotation_range=30,
                                   width_shift_range=0.2,
                                   height_shift_range=0.2,
                                   shear_range=0.2,
                                   zoom_range=0.2,
                                   horizontal_flip=True)

test_datagen = ImageDataGenerator(rescale=1./255)

In [ ]:
train_generator = train_datagen.flow_from_dataframe(train_df,
                                                    directory=folder_path,
                                                    x_col='img',
                                                    y_col=['age','gender'],
                                                    target_size=(200,200),
                                                    class_mode='multi_output')

test_generator = test_datagen.flow_from_dataframe(test_df,
                                                    directory=folder_path,
                                                    x_col='img',
                                                    y_col=['age','gender'],
                                                    target_size=(200,200),
                                                  class_mode='multi_output')

In [ ]:
from keras.applications.resnet50 import ResNet50
from keras.layers import *
from keras.models import Model

In [ ]:
resnet = ResNet50(include_top=False, input_shape=(200,200,3))

In [ ]:
resnet = ResNet50(include_top=False, input_shape=(200,200,3))

resnet.trainable=False

output = resnet.layers[-1].output

flatten = Flatten()(output)

dense1 = Dense(512, activation='relu')(flatten)
dense2 = Dense(512,activation='relu')(flatten)

dense3 = Dense(512,activation='relu')(dense1)
dense4 = Dense(512,activation='relu')(dense2)

output1 = Dense(1,activation='linear',name='age')(dense3)
output2 = Dense(1,activation='sigmoid',name='gender')(dense4)

In [ ]:
model = Model(inputs=resnet.input,outputs=[output1,output2])

In [ ]:
model.compile(optimizer='adam', loss={'age': 'mae', 'gender': 'binary_crossentropy'}, metrics={'age': 'mae', 'gender': 'accuracy'},loss_weights={'age':1,'gender':99})

In [ ]:
model.fit(train_generator, batch_size=32, epochs=10, validation_data=test_generator)